# 201 · Encode/decode cost playground

Companion to [Encode/decode cost](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/201/encode-decode-cost/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/201/encode_decode_cost.ipynb)

Cost is tokenize / numeric convert / allocate / copy—not the slogan “JSON is slow.” Wire shapes, a decimal-vs-binary microbench, then payload-shape stress.

> **Honesty:** timings are illustrative and machine-dependent. Suite [Results](https://leo-gan.github.io/GLD.SerializerBenchmark/) own rankings.


In [ ]:
import json
import struct
import timeit
from statistics import median

RECORD = {
    "id": 42,
    "temp_c": 21.5,
    "label": "sensor-7",
}



## Same logical value, different wire shapes

JSON (names + decimal text) vs fixed LE toy vs tagged sketch. Binary forms should be shorter; hex shows where metadata lives.


In [ ]:
def hex_bytes(b: bytes) -> str:
    return " ".join(f"{x:02x}" for x in b)


json_bytes = json.dumps(RECORD, separators=(",", ":")).encode("utf-8")
# layout: id u32 LE, temp f64 LE, label len u8 + utf-8 (toy, not a real format)
label = RECORD["label"].encode("utf-8")
fixed = struct.pack("<IdB", RECORD["id"], RECORD["temp_c"], len(label)) + label


def encode_varint(u: int) -> bytes:
    out = bytearray()
    while u > 0x7F:
        out.append((u & 0x7F) | 0x80)
        u >>= 7
    out.append(u & 0x7F)
    return bytes(out)


# sketch: field tags as single bytes + varint id + f64 + len-string (still toy)
sketch = (
    b"\x01" + encode_varint(RECORD["id"])
    + b"\x02" + struct.pack("<d", RECORD["temp_c"])
    + b"\x03" + bytes([len(label)]) + label
)

rows = [
    ("JSON text", json_bytes),
    ("Fixed LE toy", fixed),
    ("Tagged binary sketch", sketch),
]
print(f"{'form':22} {'nbytes':>6}  hex (truncated)")
for name, b in rows:
    print(f"{name:22} {len(b):6}  {hex_bytes(b)[:60]}{'…' if len(b) > 20 else ''}")



## Decimal digit work vs binary load

`float("21.5")` vs `struct.unpack` of IEEE bytes. Treat the ratio as a teaching signal only—not a product ranking.


In [ ]:
def parse_json_number_many(n: int = 50_000) -> float:
    total = 0.0
    s = "21.5"
    for _ in range(n):
        total += float(s)  # stand-in for decimal conversion cost
    return total


def load_binary_float_many(n: int = 50_000) -> float:
    raw = struct.pack("<d", 21.5)
    total = 0.0
    for _ in range(n):
        total += struct.unpack("<d", raw)[0]
    return total


t_json = median(timeit.repeat(parse_json_number_many, number=1, repeat=5))
t_bin = median(timeit.repeat(load_binary_float_many, number=1, repeat=5))
print(f"median wall time for 50k conversions — decimal float(): {t_json*1e3:.2f} ms")
print(f"median wall time for 50k conversions — struct unpack:   {t_bin*1e3:.2f} ms")
print("Ratio (illustrative only):", round(t_json / t_bin, 2) if t_bin else "n/a")



## Payload shape stress

Many small JSON objects vs one dense record: expect more size and slower round-trips when the graph is pointer-rich.


In [ ]:
many_small = [{"k": i, "v": i * 0.1} for i in range(500)]
one_dense = {"ks": list(range(500)), "vs": [i * 0.1 for i in range(500)]}

b_many = json.dumps(many_small, separators=(",", ":")).encode()
b_one = json.dumps(one_dense, separators=(",", ":")).encode()
print("JSON size many small objects:", len(b_many))
print("JSON size one dense record:  ", len(b_one))


def roundtrip(data):
    return json.loads(json.dumps(data))


t_many = median(timeit.repeat(lambda: roundtrip(many_small), number=20, repeat=3))
t_one = median(timeit.repeat(lambda: roundtrip(one_dense), number=20, repeat=3))
print(f"round-trip median (20×) many-small: {t_many*1e3:.2f} ms")
print(f"round-trip median (20×) one-dense:  {t_one*1e3:.2f} ms")



## Takeaways

1. Cost centers: tokenize, convert, allocate, copy.
2. Implementation and payload shape beat brand-name claims.
3. Rank libraries with suite Results on this harness.

**Next:** [Self-describing vs schema](./self_describing_vs_schema.ipynb)
